# MDRG AI Dialog — free GPU server on Google Colab

No good GPU at home? This notebook turns a **free Google Colab GPU** into an
[Ollama](https://ollama.com) server that the [MDRG AI Dialog mod](https://github.com/StLyn4/MdrgAiDialog) can talk to over the internet.

**How to use it (no technical knowledge needed):**

1. In the menu click **Runtime → Change runtime type** and make sure a **GPU** (e.g. T4) is selected.
2. Click **Runtime → Run all** and wait. The last step prints a `https://….trycloudflare.com` address.
3. Copy that address into the mod config (or the installer) as shown at the bottom.
4. **Keep this browser tab open while you play.** Closing it shuts the server down.

> ⚠️ Free Colab sessions stop after a while (usually ~90 minutes idle, 12 hours max).
> When that happens, just run the notebook again — you will get a **new** address and
> need to update the mod config once more.


## 1. Check that we actually got a GPU


In [ ]:
!nvidia-smi || echo 'No GPU! Use Runtime -> Change runtime type -> GPU'


## 2. Install Ollama


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh


## 3. Start the Ollama server


In [ ]:
import os, subprocess, time, urllib.request

env = os.environ.copy()
env['OLLAMA_HOST'] = '0.0.0.0:11434'   # accept connections from the tunnel
env['OLLAMA_ORIGINS'] = '*'            # allow requests coming through the tunnel
env['OLLAMA_KEEP_ALIVE'] = '-1'        # keep the model loaded between messages

ollama_log = open('/content/ollama.log', 'w')
ollama_proc = subprocess.Popen(['ollama', 'serve'], env=env,
                               stdout=ollama_log, stderr=subprocess.STDOUT)

for _ in range(30):
    try:
        urllib.request.urlopen('http://localhost:11434/api/version', timeout=2)
        print('Ollama server is running')
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('Ollama did not start, check /content/ollama.log')


## 4. Download the model

The default is the model the mod ships with. To use your own (for example a
Jun finetune published on Ollama or Hugging Face), change `MODEL` below —
any `ollama pull`-able name works, including `hf.co/user/repo-GGUF` names.


In [ ]:
MODEL = 'hf.co/roleplaiapp/MN-12B-Mag-Mell-R1-Q4_K_M-GGUF'  # <-- change me if you want

!ollama pull {MODEL}
!ollama list


## 5. Open a tunnel to the internet

We use a free Cloudflare *quick tunnel* — no account or signup needed.
The cell below prints the public address of your server.


In [ ]:
!curl -fL -o /usr/local/bin/cloudflared \
    https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared


In [ ]:
import re, subprocess, threading, queue

tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:11434',
     '--http-host-header', 'localhost:11434', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

public_url = None
for line in tunnel_proc.stdout:
    match = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    raise RuntimeError('Tunnel did not start, re-run this cell')

print('=' * 60)
print('YOUR SERVER ADDRESS (copy the whole line below):')
print()
print(f'    {public_url}/v1')
print()
print('=' * 60)
print()
print('Put this into <GameFolder>/UserData/MdrgAiDialog.cfg:')
print()
print('    [General]')
print('    UsedProvider = "Ollama"')
print()
print('    [Ollama]')
print(f'    ApiUrl = "{public_url}/v1"')
print(f'    Model = "{MODEL}"')
print()
print('Or run the installer on your gaming PC:')
print()
print(f'    .\\install.ps1 -OllamaUrl "{public_url}/v1" -Model "{MODEL}" -SkipOllamaInstall -SkipModelPull')


## 6. Keep the server alive

Run this last cell and **leave the tab open** while you play.
It just sits there so Colab does not consider the notebook idle.


In [ ]:
import time

print('Server is up. Leave this running while you play!')
minutes = 0
while tunnel_proc.poll() is None:
    time.sleep(60)
    minutes += 1
    print(f'Still running ({minutes} min). Address: {public_url}/v1')

print('Tunnel stopped — re-run the notebook to get a new address.')
